In [98]:
import pandas as pd
import re

In [99]:
puumala = pd.read_csv('../data/raw/puumala_all.csv')

In [100]:
puumala.shape

(3697, 28)

In [101]:
pd.set_option('display.max_colwidth', None)

In [102]:
trash_words = 'UNVERIFIED|METHODS'

In [103]:
trash_mask = puumala['GenBank_Title'].str.contains(trash_words, case=False, na=False, regex=True)

In [104]:
trash_rows = puumala[trash_mask]

In [107]:
print(f"Broj pronađenih redova sa 'UNVERIFIED' ili 'METHODS': {len(trash_rows)}")

Broj pronađenih redova sa 'UNVERIFIED' ili 'METHODS': 16


In [108]:
display(trash_rows[['Accession', 'GenBank_Title','Segment']])

,Accession,GenBank_Title,Segment
421,OQ032667.1,UNVERIFIED: Puumala orthohantavirus isolate 99 sequence,NaN
422,OQ032668.1,UNVERIFIED: Puumala orthohantavirus isolate 101 sequence,NaN
423,OQ032669.1,UNVERIFIED: Puumala orthohantavirus isolate 100 sequence,NaN
424,OQ032670.1,UNVERIFIED: Puumala orthohantavirus isolate 03 sequence,NaN
425,OQ032671.1,UNVERIFIED: Puumala orthohantavirus isolate 31 sequence,NaN
426,OQ032672.1,UNVERIFIED: Puumala orthohantavirus isolate 29 sequence,NaN
427,OQ032673.1,UNVERIFIED: Puumala orthohantavirus isolate 30 sequence,NaN
2738,HV984404.1,JP 2011191322-A/2: METHODS AND REAGENTS FOR DIAGNOSING HANTAVIRUS INFECTION,NaN
2739,HV984410.1,JP 2011191322-A/8: METHODS AND REAGENTS FOR DIAGNOSING HANTAVIRUS INFECTION,NaN
2740,HV984416.1,JP 2011191322-A/14: METHODS AND REAGENTS FOR DIAGNOSING HANTAVIRUS INFECTION,NaN


Izbacene su neverifikovane sekvencce, jer nemaju oznaku segmenta, niti definisan gen niti protein u nazivu, pa bi predstavljale samo sum u analizi. 
Patenti i metode su vestacke vijagnosticke metode ne predstavljaju odgovarajuce uzorke pa su uklnonjeni.

In [109]:
puumala = puumala[~trash_mask]

In [110]:
puumala['Segment'].value_counts(dropna=False)

Segment
S         1671
L          843
M          724
NaN        376
Small       31
Medium      22
Large        8
G2           2
G1           1
small        1
medium       1
large        1
Name: count, dtype: int64

Ujednačavanje slovnih naziva i G1/G2 oznaka u standardne S, M, L

In [111]:
puumala['Segment'] = puumala['Segment'].replace({
    'Small': 'S',
    'small': 'S',
    'Medium': 'M',
    'medium': 'M',
    'Large': 'L',
    'large': 'L',
    'G1' : 'M',
    'G2' : 'M'
})

In [112]:
puumala['Segment'].value_counts(dropna=False)

Segment
S      1703
L       852
M       750
NaN     376
Name: count, dtype: int64

In [113]:
display(puumala[puumala['Segment'].isna()][['Accession', 'GenBank_Title']].head(20))

,Accession,GenBank_Title
6,PQ867783.1,"Orthohantavirus puumalaense isolate 770_2019_PUUV_HU nucleocapsid protein gene, partial cds"
7,PQ867785.1,"Orthohantavirus puumalaense isolate 1940_2020_PUUV_HU nucleocapsid protein gene, partial cds"
8,PQ867788.1,"Orthohantavirus puumalaense isolate 2013_2021_PUUV_HU nucleocapsid protein gene, partial cds"
9,PQ867789.1,"Orthohantavirus puumalaense isolate 653_2023_PUUV_HU nucleocapsid protein gene, partial cds"
10,PV276136.1,"Orthohantavirus puumalaense isolate Cr00-18 RNA-dependent RNA polymerase gene, complete cds"
11,PV276137.1,"Orthohantavirus puumalaense isolate Cr13-1 RNA-dependent RNA polymerase gene, complete cds"
12,PV276138.1,"Orthohantavirus puumalaense isolate Cr15-1 RNA-dependent RNA polymerase gene, complete cds"
13,PV276139.1,"Orthohantavirus puumalaense isolate Cr15-13 RNA-dependent RNA polymerase gene, complete cds"
14,PV276140.1,"Orthohantavirus puumalaense isolate Cr15-15 RNA-dependent RNA polymerase gene, complete cds"
15,PV276141.1,"Orthohantavirus puumalaense isolate Cr16-3 RNA-dependent RNA polymerase gene, complete cds"


In [114]:
is_nan_segment = puumala['Segment'].isna()

In [115]:
puumala.loc[ is_nan_segment &
    puumala['GenBank_Title'].str.contains(
        'nucleocapsid',
        case=False,
        na=False
    ),
    'Segment'
] = 'S'

puumala.loc[ is_nan_segment &
    puumala['GenBank_Title'].str.contains(
        'glycoprotein',
        case=False,
        na=False
    ),
    'Segment'
] = 'M'

puumala.loc[ is_nan_segment &
    puumala['GenBank_Title'].str.contains(
        'RNA polymerase',
        case=False,
        na=False
    ),
    'Segment'
] = 'L'

In [116]:
puumala['Segment'].value_counts(dropna = False)

Segment
S      2022
L       874
M       783
NaN       2
Name: count, dtype: int64

In [117]:
display(puumala[puumala['Segment'].isna()][['Accession', 'GenBank_Title']])

,Accession,GenBank_Title
3286,AM695639.1,"Puumala virus partial M gene for Gc protein, strain PUU/Mignovillard/CgY02/2005, genomic RNA"
3687,M29979.1,"Puumala virus CG1820 virus M genome segment, complete cds"


Prvi red sadrzi oznaku M gene i Gc protein, sto znaci da pripada M segmentu. Takodje i drugi red pripada M segmentu.

In [118]:
m_accessions = ['AM695639.1', 'M29979.1']
puumala.loc[puumala['Accession'].isin(m_accessions), 'Segment'] = 'M'

In [119]:
puumala['Segment'].value_counts(dropna=False)

Segment
S    2022
L     874
M     785
Name: count, dtype: int64

In [120]:
cols = ['Accession','Segment',  'Nuc_Completeness', 'Species']

In [121]:
all_sequences = puumala[cols]
all_sequences.to_csv('../data/processed_sequences/all_sequences/puumala_all.csv')

In [122]:
complete_sequences = all_sequences[all_sequences['Nuc_Completeness'] == 'complete'].copy()

In [123]:
complete_sequences.to_csv('../data/processed_sequences/complete_sequences/puumala_complete.csv')